# Fábrica dos Cortes

Cortes verticais para TikTok, Reels e Shorts + compilado **Melhores Momentos** para YouTube.

**Notebook v4**

### Como usar
1. Rode **1. Instalar** e espere aparecer `AMBIENTE OK`.
2. Rode **2. Iniciar**.
3. Clique no link `.gradio.live` que aparecer.
4. Na tela do app, cole sua chave Gemini e gere os cortes.

Se o download do YouTube falhar, use **3. Cookies**.
Se qualquer coisa falhar, rode **4. Diagnóstico** e me mande a saída.


In [ ]:
#@title 🛠️ 1. Instalar (5 a 10 minutos)
import os
import shutil
import subprocess
import sys

REPO = "https://github.com/bazotepedro-oss/forjaviral.git"
ROOT = "/content/forjaviral"
PY = os.path.join(ROOT, ".venv", "bin", "python")

print("=" * 60)
print("FABRICA DOS CORTES - INSTALADOR v4")
print("=" * 60)


def rodar(comando, etapa, obrigatorio=True):
    print(f"\n>> {etapa}")
    resultado = subprocess.run(comando, shell=True, text=True, capture_output=True)
    if resultado.stdout:
        print(resultado.stdout[-2500:])
    if resultado.returncode != 0:
        print(resultado.stderr[-4000:])
        if obrigatorio:
            raise SystemExit(f"FALHOU: {etapa}. Copie as ultimas linhas acima.")
        print(f"[AVISO] {etapa} falhou, seguindo em frente.")
        return False
    return True


print("\n1/7  Limpando instalacao anterior...")
os.chdir("/content")
if os.path.exists(ROOT):
    shutil.rmtree(ROOT)

print("2/7  Baixando o codigo mais recente...")
rodar(f"git clone --depth 1 {REPO} {ROOT}", "git clone")
os.chdir(ROOT)

print("3/7  Dependencias do sistema...")
rodar(
    "apt-get update -qq && apt-get install -y -qq ffmpeg xvfb libgl1 libglib2.0-0",
    "apt-get",
    obrigatorio=False,
)
if shutil.which("ffmpeg") is None:
    raise SystemExit("ffmpeg nao encontrado. Reinicie o ambiente do Colab e rode de novo.")

print("4/7  Python 3.11 e ambiente virtual...")
rodar(f"{sys.executable} -m pip install -q uv", "instalar uv")
rodar("uv python install 3.11", "baixar Python 3.11")
rodar("uv venv --python 3.11 .venv", "criar ambiente virtual")

print("5/7  PyTorch na mesma versao do torchaudio (CUDA 12.8)...")
rodar(
    "uv pip install --python .venv torch==2.8.0 torchaudio==2.8.0 "
    "--index-url https://download.pytorch.org/whl/cu128",
    "torch + torchaudio",
)

print("6/7  Bibliotecas de IA...")
rodar("uv pip install --python .venv -r requirements.txt", "requirements.txt")
rodar("uv pip install --python .venv yt-dlp pytubefix", "yt-dlp e pytubefix")

print("7/7  Testando o ambiente (inclusive o WhisperX)...")
verificacao = """
import torch, torchaudio, gradio, yt_dlp, pytubefix
print('torch      ', torch.__version__)
print('torchaudio ', torchaudio.__version__)
print('gradio     ', gradio.__version__)
print('CUDA       ', torch.cuda.is_available())
base_torch = torch.__version__.split('+')[0]
base_audio = torchaudio.__version__.split('+')[0]
if base_torch != base_audio:
    raise SystemExit('torch e torchaudio em versoes diferentes: ' + base_torch + ' x ' + base_audio)
import whisperx
print('whisperx   ', getattr(whisperx, '__version__', 'ok'))
print('AMBIENTE OK')
"""
open("_check_env.py", "w", encoding="utf-8").write(verificacao)
rodar(f"{PY} _check_env.py", "verificacao final")

os.makedirs(os.path.join(ROOT, "VIRALS"), exist_ok=True)
print("\n" + "=" * 60)
print("INSTALACAO CONCLUIDA. Rode a celula 2.")
print("=" * 60)


In [ ]:
#@title 🚀 2. Iniciar
import os
import re
import subprocess
import sys

ROOT = "/content/forjaviral"
PY = os.path.join(ROOT, ".venv", "bin", "python")

if not os.path.exists(PY):
    raise SystemExit("Rode a celula 1 (Instalar) primeiro.")

os.chdir(ROOT)
os.environ["DISPLAY"] = ":1.0"
os.environ["MPLBACKEND"] = "Agg"
os.environ["PYTHONUNBUFFERED"] = "1"
os.makedirs("VIRALS", exist_ok=True)
os.makedirs("logs", exist_ok=True)

subprocess.Popen(
    "Xvfb :1 -screen 0 2560x1440x8",
    shell=True,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

print("Abrindo a interface. O link aparece destacado abaixo.\n")

log_path = os.path.join("logs", "colab_app.log")
processo = subprocess.Popen(
    [PY, "webui/app.py", "--colab"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    errors="replace",
)

link = None
with open(log_path, "w", encoding="utf-8", errors="replace") as log:
    for linha in processo.stdout:
        sys.stdout.write(linha)
        sys.stdout.flush()
        log.write(linha)
        if link is None:
            achado = re.search(r"https://[a-zA-Z0-9.-]+\.gradio\.live", linha)
            if achado:
                link = achado.group(0)
                print("\n" + "=" * 64)
                print("LINK DA FABRICA DOS CORTES:")
                print(link)
                print("=" * 64 + "\n")

processo.wait()

print("\nO aplicativo encerrou com codigo:", processo.returncode)
if link:
    print("Link desta execucao:", link)
else:
    print("Nenhum link .gradio.live apareceu. Ultimas linhas do log:")
    with open(log_path, encoding="utf-8", errors="replace") as log:
        print("\n".join(log.read().splitlines()[-30:]))


In [ ]:
#@title 🍪 3. Opcional: cookies do YouTube
# Use somente se o download falhar com "Sign in to confirm you're not a bot".
# Pare a celula 2, rode esta e depois rode a celula 2 de novo.
import os
import shutil

try:
    from google.colab import files
except ImportError:
    files = None

DESTINO = "/content/forjaviral/cookies.txt"

print("Como obter o cookies.txt:")
print("1. Instale a extensao 'Get cookies.txt LOCALLY' no Chrome ou Edge.")
print("2. Abra o YouTube logado.")
print("3. Exporte o arquivo cookies.txt.")
print("4. Envie abaixo.\n")

if files is None:
    print("Esta celula so funciona no Colab.")
else:
    enviados = files.upload()
    salvo = False
    for nome in enviados:
        if not nome.lower().endswith(".txt"):
            continue
        shutil.copyfile(nome, DESTINO)
        shutil.copyfile(nome, "/content/cookies.txt")
        print(f"Salvo em: {DESTINO} ({os.path.getsize(DESTINO)} bytes)")
        with open(DESTINO, "r", encoding="utf-8", errors="ignore") as fh:
            primeira = fh.readline().strip()
        if "netscape" in primeira.lower() or "\t" in primeira:
            print("Formato reconhecido.")
        else:
            print("[AVISO] O arquivo nao parece um cookies.txt do Netscape.")
        salvo = True
    if salvo:
        print("\nAgora rode a celula 2 de novo.")
    else:
        print("\nNenhum arquivo .txt enviado.")


In [ ]:
#@title 🔎 4. Diagnostico (use se algo falhar)
import glob
import os

prints = []

for caminho in sorted(glob.glob("/content/forjaviral/logs/render_*.log"), key=os.path.getmtime)[-2:]:
    prints.append(("RENDER", caminho, open(caminho, encoding="utf-8", errors="replace").read()))

for caminho in ["/content/forjaviral/logs/colab_app.log"]:
    if os.path.exists(caminho):
        prints.append(("APP", caminho, open(caminho, encoding="utf-8", errors="replace").read()))

if not prints:
    print("Nenhum log encontrado em /content/forjaviral/logs")
else:
    for tipo, caminho, texto in prints:
        linhas = [l for l in texto.splitlines() if l.strip()]
        print("=" * 70)
        print(tipo, caminho)
        print("=" * 70)
        for linha in linhas[-40:]:
            print(linha)


# Suporte

**WhatsApp:** [+55 18 98122-3199](https://wa.me/5518981223199)  
**Grupo de alunos:** [entrar](https://chat.whatsapp.com/G0yGg1Rl0K3J8OUEChfc4s)  
**Site:** https://fabricadecortes.com


`1.3.28`

Fábrica dos Cortes — projeto independente.
